# Actor Clustering - Spectral

This notebook implements the actor-level clustering pipeline for one model using the shared preprocessing and graph construction logic.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.spectral import run_spectral
from src.preprocessing.pipeline import prepare_common_actor_data
from src.utils.config import RESULTS_DIR, SEED
from src.utils.io_utils import save_json, save_results_csv

MAX_NODES = 2000
N_CLUSTERS = 8
common = prepare_common_actor_data(max_nodes=MAX_NODES)
assignments, metrics = run_spectral(common["graph"], n_clusters=N_CLUSTERS, seed=SEED)
result_dir = RESULTS_DIR / "spectral"
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
print(metrics)

/home/vinhnguyen/Workspace/Personal/elliptic-actor-graph/.venv/lib/python3.13/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


{'method': 'spectral', 'n_nodes': 2000, 'n_edges': 1031, 'n_clusters': 8, 'runtime_seconds': 0.29105734399490757, 'seed': 42, 'requested_clusters': 8}


In [2]:
from src.clustering.evaluation import evaluate_clusters
from src.clustering.visualization import save_cluster_figures, save_graph_partition_figure

labels = common["actor_df"].set_index("actor_id")["class"].reindex(assignments["actor_id"])
metrics.update(evaluate_clusters(assignments, labels=labels))
metrics["research_questions"] = [
    "How sensitive is feature-free spectral clustering to the selected number of communities?",
    "Does spectral partitioning reveal structure that modularity optimization misses?",
]
save_json(metrics, result_dir / "metrics.json")
save_cluster_figures(assignments, result_dir, title="Spectral actor communities")
save_graph_partition_figure(common["graph"], assignments, result_dir / "graph_partition.png", "Spectral partition on AddrAddr graph")
print("Research interpretation metrics:", {key: metrics[key] for key in ("silhouette", "ari", "nmi", "ami")})

Research interpretation metrics: {'silhouette': None, 'ari': -0.006893064396372314, 'nmi': 0.0011240376797839147, 'ami': -0.0033497168315673445}
